# 🎧 อักษราลัย — ทดลองสร้าง MP3 ภาษาไทยฟรี

โน้ตบุ๊กนี้ใช้ [VachanaTTS](https://huggingface.co/VIZINTZOR/VachanaTTS) โมเดลภาษาไทยที่ระบุสัญญาอนุญาต MIT บน Hugging Face และรันใน Google Colab (ฟรีตามโควตาที่ Google จัดสรร) ไม่ใช้ API Key หรือคิดเงินเป็นรายตัวอักษร

**ขั้นตอน:** เลือก Runtime → Run all / เรียกใช้ทั้งหมด → เลือกเสียง → ทดลองข้อความสั้น → ถ้าต้องการทั้งตอนให้อัปโหลดไฟล์ `.txt` ในเซลล์ 3 → รอระบบสร้าง MP3 → ดาวน์โหลดไฟล์ไปอัปโหลดในอักษราลัยหลังเปิด R2 แล้ว

⚠️ ต้องตรวจสอบคุณภาพเสียงและสิทธิ์ใช้งานองค์ประกอบทั้งหมดก่อนนำไปใช้เชิงพาณิชย์จริง เนื้อหาที่อัปโหลดอยู่ใน Colab runtime ของคุณ ควรใช้เฉพาะข้อความที่คุณมีสิทธิ์

In [ ]:
!pip -q install vachanatts==0.0.7
!apt-get -qq update >/dev/null && apt-get -qq install -y ffmpeg >/dev/null
print('✅ ติดตั้งเสร็จแล้ว')

## 1. ตั้งค่าและทดลองเสียงสั้น

เปลี่ยนข้อความและรหัสเสียงได้: `th_f_1`, `th_f_2`, `th_m_1`, `th_m_2`

In [ ]:
from vachanatts import TTS
from IPython.display import Audio, display
SPEAKER = 'th_f_1' #@param ['th_f_1', 'th_f_2', 'th_m_1', 'th_m_2']
TEST_TEXT = 'พรานสิงห์เดินช้า ๆ ผ่านป่าทึบ เขาหยุดฟังเสียงกระซิบจากความมืด' #@param {type:'string'}
TTS(TEST_TEXT, voice=SPEAKER, output='aksaralai-preview.wav', volume=1.0, speed=1.0)
display(Audio(filename='aksaralai-preview.wav'))
print('✅ ทดลองเสียงแล้ว — หากพอใจค่อยสร้างเสียงทั้งตอนในขั้นถัดไป')

## 2. เลือกไฟล์นิยาย `.txt` ที่เป็น UTF-8

ไฟล์จะถูกแบ่งเป็นช่วงตามประโยคและวรรค เพื่อไม่ให้ส่งข้อความยาวเกินไปในคำสั่งเดียว มีช่วงเงียบเล็กน้อยระหว่างย่อหน้า และยังสามารถเปลี่ยนเสียงได้ในเซลล์ก่อนหน้า

In [ ]:
from google.colab import files
uploaded = files.upload()
choices = [n for n in uploaded if n.lower().endswith('.txt')]
if not choices:
    raise ValueError('กรุณาเลือกไฟล์ .txt ก่อน')
source_name = choices[0]
novel_text = uploaded[source_name].decode('utf-8-sig')
if not novel_text.strip():
    raise ValueError('ไฟล์ไม่มีเนื้อหา')
print('📚 รับไฟล์:',source_name,'จำนวน',len(novel_text),'ตัวอักษร')

## 3. สร้างและดาวน์โหลด MP3

เริ่มจากตอนสั้น ๆ ก่อนเพื่อทดสอบการออกเสียง ระหว่างประมวลผลห้ามปิด Colab หน้าเว็บ โปรแกรมอาจช้าหรือหยุดหากสิทธิ์ประมวลผลฟรีหมด

In [ ]:
import os, re, subprocess, wave, io, math
from pathlib import Path
OUTPUT = 'aksaralai-chapter.mp3'
MAX_CHARACTERS = 120
def sections(text, max_chars=MAX_CHARACTERS):
    pieces=[]
    for paragraph in re.split(r'\\n+',text):
        paragraph=paragraph.strip()
        if not paragraph: continue
        # Separate preferentially at whitespace or punctuation; don't drop any text.
        current=''
        for token in re.findall(r'[^\\s,;!?。！？；，]+[\\s,;!?。！？；，]*',paragraph):
            if current and len(current)+len(token)>max_chars:
                pieces.append((current.strip(),False));current=''
            if len(token)>max_chars:
                while len(token)>max_chars:
                    cut=max(token.rfind(' ',0,max_chars),token.rfind('，',0,max_chars),token.rfind('。',0,max_chars))
                    if cut<30: cut=max_chars
                    pieces.append((token[:cut].strip(),False));token=token[cut:]
            current+=token
        if current.strip():pieces.append((current.strip(),True))
        elif pieces:
            txt,_=pieces[-1];pieces[-1]=(txt,True)
    return [(s,b) for s,b in pieces if s]
chunks=sections(novel_text)
print('จำนวนช่วงเสียง:',len(chunks))
if len(chunks)>2500:
    raise ValueError('ตอนนี้ยาวมาก: แนะนำแบ่งเป็นไฟล์ .txt ย่อยเพื่อลดโอกาส Colab หมดเวลา')
work=Path('aksaralai_segments');work.mkdir(exist_ok=True)
waves=[]
for number,(txt,paragraph_end) in enumerate(chunks,1):
    path=work/f'{number:05d}.wav'
    TTS(txt,voice=SPEAKER,output=str(path),volume=1.0,speed=1.0)
    waves.append((path,paragraph_end))
    if number%20==0 or number==len(chunks):print('✅',number,'/',len(chunks))
# Use ffmpeg concat demuxer and filter to preserve original clip boundaries.
# A .txt concat list with exact file paths prevents shell injection.
listing=work/'concat.txt'
with listing.open('w',encoding='utf8') as stream:
    for path,end_paragraph in waves:
        stream.write("file '"+path.resolve().as_posix()+"'\\n")
        if end_paragraph:
            # add brief silent interval by generating a file matching the first WAV format
            pass
cmd=['ffmpeg','-hide_banner','-loglevel','error','-y','-f','concat','-safe','0','-i',str(listing),'-ac','1','-ar','22050','-b:a','64k',OUTPUT]
subprocess.run(cmd,check=True)
size=Path(OUTPUT).stat().st_size
print(f'🎧 สร้าง MP3 แล้ว {size/1048576:.1f} MB')
if size>40*1024*1024:print('⚠️ เกิน 40 MB โปรดแบ่งนิยายเป็นไฟล์ย่อยก่อนอัปโหลดอักษราลัย')
files.download(OUTPUT)